# Cityscapes Semantic Segmentation Benchmark

**SegFormer-B2 (Transformer) vs DeepLabV3+ (CNN)**

This notebook trains, evaluates, and exports both models on the Cityscapes dataset.


> **Runtime**: Go to `Runtime → Change runtime type → T4 GPU` before starting.

---
## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
# Verify GPU is available
!nvidia-smi

In [ ]:
# Clone the repository
# Replace with your actual GitHub repo URL
!git clone https://github.com/YOUR_USERNAME/cityscapes-benchmark.git /content/cityscapes-benchmark

# If you already cloned, just pull latest
# !cd /content/cityscapes-benchmark && git pull

In [ ]:
%cd /content/cityscapes-benchmark/ml-pipeline

# Install all dependencies
!pip install -q -r requirements.txt

In [ ]:
# Quick sanity check — all imports should succeed
import torch
import torchvision
import transformers
import albumentations

print(f"PyTorch    : {torch.__version__}")
print(f"TorchVision: {torchvision.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA       : {torch.cuda.is_available()} → {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

---
## 2. Download Cityscapes Dataset

We need two components:
- **leftImg8bit** (~11 GB) — RGB street-scene images → downloaded directly here
- **gtFine** (~241 MB) — Ground-truth labels → uploaded from Google Drive

### 2a. Set your Cityscapes credentials

> Register at [cityscapes-dataset.com/register](https://www.cityscapes-dataset.com/register/) if you don't have an account.

In [ ]:
import os

# ╔══════════════════════════════════════════════════════╗
# ║  CHANGE THESE TO YOUR CITYSCAPES CREDENTIALS        ║
# ╚══════════════════════════════════════════════════════╝
os.environ['CITYSCAPES_USERNAME'] = 'your_email@example.com'   # ← CHANGE
os.environ['CITYSCAPES_PASSWORD'] = 'your_password'            # ← CHANGE

### 2b. Upload gtFine from Google Drive

Upload `gtFine_trainvaltest.zip` to your Google Drive first, then run:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Extract gtFine labels from Google Drive
# Adjust the path if your zip is in a different location
!mkdir -p /content/cityscapes
!unzip -q "/content/drive/MyDrive/gtFine_trainvaltest.zip" -d /content/cityscapes/

print("✓ gtFine extracted")

### 2c. Download leftImg8bit directly from Cityscapes server

In [ ]:
%cd /content/cityscapes-benchmark/ml-pipeline

# This downloads ~11GB directly to the Colab instance
# Takes ~15-30 minutes depending on network speed
!python colab_setup.py --skip-deps

### 2d. Verify dataset

In [ ]:
import os
from pathlib import Path

IMAGES_ROOT = "/content/cityscapes/leftImg8bit"
LABELS_ROOT = "/content/cityscapes/gtFine"

for split in ["train", "val"]:
    img_dir = Path(IMAGES_ROOT) / split
    lbl_dir = Path(LABELS_ROOT) / split

    n_images = len(list(img_dir.rglob("*_leftImg8bit.png"))) if img_dir.exists() else 0
    n_labels = len(list(lbl_dir.rglob("*_gtFine_labelIds.png"))) if lbl_dir.exists() else 0

    status = "✓" if n_images > 0 and n_labels > 0 else "✗"
    print(f"  {status} {split}: {n_images} images, {n_labels} labels")

print(f"\nIMAGES_ROOT = '{IMAGES_ROOT}'")
print(f"LABELS_ROOT = '{LABELS_ROOT}'")

---
## 3. Train SegFormer-B2

Fine-tunes `nvidia/segformer-b2` pre-trained on ImageNet-1k.

| Param | Value |
|-------|-------|
| Backbone | MiT-B2 (Transformer) |
| Parameters | ~27M |
| Batch size | 4 |
| Learning rate | 6e-5 |
| Epochs | 50 |
| Mixed precision | ✓ |
| Expected mIoU | 0.78–0.83 |
| Expected time | ~2h on T4 |

In [ ]:
%cd /content/cityscapes-benchmark/ml-pipeline

!python -m src.train \
    --model segformer \
    --images-root /content/cityscapes/leftImg8bit \
    --labels-root /content/cityscapes/gtFine \
    --epochs 50 \
    --batch-size 4 \
    --lr 6e-5 \
    --mixed-precision \
    --output-dir /content/runs/segformer

---
## 4. Train DeepLabV3+

Fine-tunes `torchvision.deeplabv3_resnet101` pre-trained on COCO.

| Param | Value |
|-------|-------|
| Backbone | ResNet-101 (CNN) |
| Parameters | ~59M |
| Batch size | 4 |
| Learning rate | 1e-4 |
| Epochs | 50 |
| Mixed precision | ✓ |
| Expected mIoU | 0.73–0.78 |
| Expected time | ~3h on T4 |

In [ ]:
!python -m src.train \
    --model deeplabv3 \
    --images-root /content/cityscapes/leftImg8bit \
    --labels-root /content/cityscapes/gtFine \
    --epochs 50 \
    --batch-size 4 \
    --lr 1e-4 \
    --mixed-precision \
    --output-dir /content/runs/deeplabv3

---
## 5. Evaluate Both Models

Runs the validation set through both models and generates `metrics.json` with:
- mIoU, pixel accuracy, per-class IoU
- Latency (mean, p50, p95), FPS
- Model size, parameter count, peak GPU memory

In [ ]:
!python evaluate_local.py \
    --images-root /content/cityscapes/leftImg8bit \
    --labels-root /content/cityscapes/gtFine \
    --segformer-checkpoint /content/runs/segformer/best_model.pth \
    --deeplabv3-checkpoint /content/runs/deeplabv3/best_model.pth \
    --output /content/metrics.json

In [ ]:
# Preview the metrics
import json

with open('/content/metrics.json') as f:
    metrics = json.load(f)

for model_key, info in metrics['models'].items():
    print(f"\n{'='*50}")
    print(f"  {info['name']}")
    print(f"{'='*50}")
    print(f"  mIoU           : {info['mIoU']:.4f}")
    print(f"  Pixel Accuracy : {info['pixelAccuracy']:.4f}")
    print(f"  Latency (mean) : {info['latencyMs']['mean']} ms")
    print(f"  FPS            : {info['fps']}")
    print(f"  Model Size     : {info['modelSizeMB']} MB")
    print(f"  Parameters     : {info['paramCount']:,}")

---
## 6. Export to ONNX

Exports both models to ONNX format for the web portal inference engine.

In [ ]:
# Export SegFormer to ONNX
!python -m src.export \
    --model segformer \
    --checkpoint /content/runs/segformer/best_model.pth \
    --output /content/exports/segformer_b2.onnx

In [ ]:
# Export DeepLabV3 to ONNX
!python -m src.export \
    --model deeplabv3 \
    --checkpoint /content/runs/deeplabv3/best_model.pth \
    --output /content/exports/deeplabv3_resnet101.onnx

In [ ]:
# Check exported file sizes
import os

for name in ['segformer_b2.onnx', 'deeplabv3_resnet101.onnx']:
    path = f'/content/exports/{name}'
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"  ✓ {name}: {size:.1f} MB")
    else:
        print(f"  ✗ {name}: NOT FOUND")

---
## 7. Visualization

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

with open('/content/metrics.json') as f:
    data = json.load(f)

seg = data['models']['segformer']
dl  = data['models']['deeplabv3']

classes = list(seg['perClassIoU'].keys())
seg_iou = [seg['perClassIoU'][c] for c in classes]
dl_iou  = [dl['perClassIoU'][c]  for c in classes]

In [ ]:
# ─── Per-class IoU comparison bar chart ───────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))
x = np.arange(len(classes))
w = 0.35

bars1 = ax.bar(x - w/2, [v*100 for v in seg_iou], w,
               label=f"SegFormer-B2 (mIoU={seg['mIoU']:.3f})",
               color='#3b82f6', edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x + w/2, [v*100 for v in dl_iou], w,
               label=f"DeepLabV3+ (mIoU={dl['mIoU']:.3f})",
               color='#a855f7', edgecolor='white', linewidth=0.5)

ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('IoU (%)', fontsize=12)
ax.set_title('Per-Class IoU Comparison — Cityscapes Validation', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=10)
ax.legend(fontsize=11, loc='lower right')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('/content/iou_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Summary comparison table ────────────────────────────────────
summary_metrics = [
    ('mIoU',              f"{seg['mIoU']:.4f}",              f"{dl['mIoU']:.4f}"),
    ('Pixel Accuracy',    f"{seg['pixelAccuracy']:.4f}",     f"{dl['pixelAccuracy']:.4f}"),
    ('Latency (mean)',    f"{seg['latencyMs']['mean']} ms",  f"{dl['latencyMs']['mean']} ms"),
    ('Latency (p95)',     f"{seg['latencyMs']['p95']} ms",   f"{dl['latencyMs']['p95']} ms"),
    ('FPS',               f"{seg['fps']}",                   f"{dl['fps']}"),
    ('Model Size',        f"{seg['modelSizeMB']} MB",        f"{dl['modelSizeMB']} MB"),
    ('Parameters',        f"{seg['paramCount']:,}",          f"{dl['paramCount']:,}"),
    ('Peak GPU Memory',   f"{seg['peakMemoryMB']} MB",       f"{dl['peakMemoryMB']} MB"),
]

print(f"\n{'Metric':<20} {'SegFormer-B2':>20} {'DeepLabV3+':>20}")
print('─' * 62)
for metric, sv, dv in summary_metrics:
    print(f"{metric:<20} {sv:>20} {dv:>20}")

In [ ]:
# ─── Sample prediction visualization ────────────────────────────
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
import sys
sys.path.insert(0, '/content/cityscapes-benchmark/ml-pipeline')

from src.models import SegFormerModel, DeepLabV3Model
from src.dataset import NUM_CLASSES, CITYSCAPES_CLASSES

# Cityscapes colour palette (trainId order)
PALETTE = [
    (128,64,128), (244,35,232), (70,70,70), (102,102,156), (190,153,153),
    (153,153,153), (250,170,30), (220,220,0), (107,142,35), (152,251,152),
    (70,130,180), (220,20,60), (255,0,0), (0,0,142), (0,0,70),
    (0,60,100), (0,80,100), (0,0,230), (119,11,32),
]

def colorize_mask(mask_np):
    """Convert class-ID mask to RGB."""
    h, w = mask_np.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cid, color in enumerate(PALETTE):
        rgb[mask_np == cid] = color
    return rgb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load models
seg_model = SegFormerModel(num_classes=NUM_CLASSES)
seg_ckpt = torch.load('/content/runs/segformer/best_model.pth', map_location=device, weights_only=True)
if 'model_state_dict' in seg_ckpt:
    seg_model.load_state_dict(seg_ckpt['model_state_dict'])
else:
    seg_model.load_state_dict(seg_ckpt)
seg_model = seg_model.to(device).eval()

dl_model = DeepLabV3Model(num_classes=NUM_CLASSES, pretrained=False)
dl_ckpt = torch.load('/content/runs/deeplabv3/best_model.pth', map_location=device, weights_only=True)
if 'model_state_dict' in dl_ckpt:
    dl_model.load_state_dict(dl_ckpt['model_state_dict'])
else:
    dl_model.load_state_dict(dl_ckpt)
dl_model = dl_model.to(device).eval()

print("✓ Both models loaded")

In [ ]:
# Pick a random validation image
import glob, random

val_images = sorted(glob.glob('/content/cityscapes/leftImg8bit/val/**/*_leftImg8bit.png', recursive=True))
sample_path = random.choice(val_images)
print(f"Sample: {sample_path}")

# Preprocess
img = Image.open(sample_path).convert('RGB')
img_resized = img.resize((1024, 512), Image.BILINEAR)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
input_tensor = transform(img_resized).unsqueeze(0).to(device)

# Predict
with torch.no_grad():
    seg_logits = seg_model(input_tensor)
    dl_logits  = dl_model(input_tensor)

    # Upsample to input size
    seg_logits = F.interpolate(seg_logits, size=(512, 1024), mode='bilinear', align_corners=False)
    dl_logits  = F.interpolate(dl_logits,  size=(512, 1024), mode='bilinear', align_corners=False)

seg_mask = seg_logits.argmax(dim=1).squeeze().cpu().numpy()
dl_mask  = dl_logits.argmax(dim=1).squeeze().cpu().numpy()

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

axes[0].imshow(img_resized)
axes[0].set_title('Original', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(colorize_mask(seg_mask))
axes[1].set_title(f"SegFormer-B2 (mIoU={seg['mIoU']:.3f})", fontsize=14, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(colorize_mask(dl_mask))
axes[2].set_title(f"DeepLabV3+ (mIoU={dl['mIoU']:.3f})", fontsize=14, fontweight='bold')
axes[2].axis('off')

plt.suptitle('Side-by-Side Segmentation Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/segmentation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Overlay visualization ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

img_np = np.array(img_resized)

# SegFormer overlay
seg_rgb = colorize_mask(seg_mask)
overlay_seg = (img_np * 0.4 + seg_rgb * 0.6).astype(np.uint8)
axes[0].imshow(overlay_seg)
axes[0].set_title('SegFormer-B2 Overlay', fontsize=14, fontweight='bold')
axes[0].axis('off')

# DeepLabV3 overlay
dl_rgb = colorize_mask(dl_mask)
overlay_dl = (img_np * 0.4 + dl_rgb * 0.6).astype(np.uint8)
axes[1].imshow(overlay_dl)
axes[1].set_title('DeepLabV3+ Overlay', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('/content/overlay_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Download Artifacts

Download these files and place them in the web portal:
- `metrics.json` → `web-portal/public/metrics.json`
- `segformer_b2.onnx` → `web-portal/public/models/segformer_b2.onnx`
- `deeplabv3_resnet101.onnx` → `web-portal/public/models/deeplabv3_resnet101.onnx`

In [ ]:
from google.colab import files

# Download metrics (for the web dashboard)
files.download('/content/metrics.json')

# Download ONNX models (for web portal inference)
files.download('/content/exports/segformer_b2.onnx')
files.download('/content/exports/deeplabv3_resnet101.onnx')

# Download visualizations
files.download('/content/iou_comparison.png')
files.download('/content/segmentation_comparison.png')
files.download('/content/overlay_comparison.png')

In [ ]:
# Optional: also download the PyTorch checkpoints
files.download('/content/runs/segformer/best_model.pth')
files.download('/content/runs/deeplabv3/best_model.pth')

In [ ]:
# Optional: save everything to Google Drive
!mkdir -p "/content/drive/MyDrive/cityscapes-benchmark/exports"
!mkdir -p "/content/drive/MyDrive/cityscapes-benchmark/checkpoints"

!cp /content/metrics.json "/content/drive/MyDrive/cityscapes-benchmark/"
!cp /content/exports/*.onnx "/content/drive/MyDrive/cityscapes-benchmark/exports/"
!cp /content/runs/segformer/best_model.pth "/content/drive/MyDrive/cityscapes-benchmark/checkpoints/segformer_best.pth"
!cp /content/runs/deeplabv3/best_model.pth "/content/drive/MyDrive/cityscapes-benchmark/checkpoints/deeplabv3_best.pth"
!cp /content/*.png "/content/drive/MyDrive/cityscapes-benchmark/"

print("\n✓ All artifacts saved to Google Drive!")

---
## Done!

### Next steps:

1. Copy `segformer_b2.onnx` and `deeplabv3_resnet101.onnx` → `web-portal/public/models/`
2. Copy `metrics.json` → `web-portal/public/metrics.json`
3. Run the web portal: `cd web-portal && npm run dev`
4. Open http://localhost:3000 and upload a street image!

See `DEPLOYMENT.md` for production deployment (Vercel, Docker, etc).